In [ ]:
from pathlib import Path
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/tsr_samples")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/pt_samples_far")

REPLICA_EXCHANGE = True
LAM_VALUES = LAM_VALUES
INDEX_UNTIL = 6

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import gc

# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")

gc.collect()
torch.cuda.empty_cache()

from diffusers import StableDiffusion3Pipeline

# ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

if REPLICA_EXCHANGE:
	BASE_OUTPUT_DIR = PT_TSR_DIR
else:
	BASE_OUTPUT_DIR = TSR_DIR

lam_dirs = {l: BASE_OUTPUT_DIR / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
for d in lam_dirs.values():
    d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	if all((BASE_OUTPUT_DIR / f"lam{l:.3f}".replace(".", "p") / f"{idx:05d}.png").exists() for l in LAM_VALUES):
		continue
	
	generator = torch.Generator(device="cuda").manual_seed(SEED + idx)

	for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx}"):

		output_dir = lam_dirs[tsr_lam]

		if (output_dir / f"{idx:05d}.png").exists():
			continue

		images = pipe(
			prompt,
			negative_prompt="",
			num_inference_steps=N_INF_STEPS,
			guidance_scale=GUIDANCE_SCALE,
			tsr_lam=tsr_lam,
			tsr_sigma=TSR_SIGMA,
			replica_exchange=REPLICA_EXCHANGE,
			swap_algorithm=SWAP_ALGORITHM,
			generator=generator,
		).images

		out_path = output_dir / f"{idx:05d}.png"
		images[0].save(out_path, icc_profile=None)
		del images
	torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 6 prompts


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

 We tsr by 1.01 with replica exchange True
Time 1000.00 swap btwn source 0.90 and target 1.01 accept 0.960 std 0.987
Time 949.53 swap btwn source 1.06 and target 1.01 accept 0.999 std 0.939
Time 887.74 swap btwn source 0.90 and target 1.01 accept 0.951 std 0.890
Time 810.31 swap btwn source 1.06 and target 1.01 accept 0.993 std 0.842
Time 710.47 swap btwn source 0.90 and target 1.01 accept 0.949 std 0.808
Time 576.82 swap btwn source 1.06 and target 1.01 accept 0.985 std 0.805


idx=1:  67%|██████▋   | 2/3 [01:06<00:33, 33.20s/it]

 We tsr by 1.05 with replica exchange True
Time 1000.00 swap btwn source 0.90 and target 1.05 accept 0.961 std 0.987
Time 949.53 swap btwn source 1.10 and target 1.05 accept 0.999 std 0.939
Time 887.74 swap btwn source 0.90 and target 1.05 accept 0.955 std 0.890
Time 810.31 swap btwn source 1.10 and target 1.05 accept 0.992 std 0.843
Time 710.47 swap btwn source 0.90 and target 1.05 accept 0.966 std 0.811
Time 576.82 swap btwn source 1.10 and target 1.05 accept 0.987 std 0.815


idx=2:   0%|          | 0/3 [00:00<?, ?it/s]

 We tsr by 1.10 with replica exchange True
Time 1000.00 swap btwn source 0.90 and target 1.10 accept 0.960 std 0.988
Time 949.53 swap btwn source 1.15 and target 1.10 accept 0.999 std 0.940
Time 887.74 swap btwn source 0.90 and target 1.10 accept 0.958 std 0.886
Time 810.31 swap btwn source 1.15 and target 1.10 accept 0.993 std 0.829
Time 710.47 swap btwn source 0.90 and target 1.10 accept 0.983 std 0.775
Time 576.82 swap btwn source 1.15 and target 1.10 accept 0.990 std 0.741


In [ ]:
from fid import compute_sweep

compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True],
	device="cuda",
	target_indices=None,
	index_until=INDEX_UNTIL,
	pt_sr_dir = PT_TSR_DIR,
)